# Aula 7 — Bands, density of states and magnetism

Hands-on: band structure and DOS of silicon, what the Kohn–Sham eigenvalues mean (the Si atom), and ferromagnetic iron.
Run the cells in order. Every number quoted below was checked with QE 6.7 (the version `apt` installs on Colab).

## 1. Set up (every session)
Same two cells as in Aula 6: install QE (about 1 min), then open the course folder in your Drive.
The second cell now also runs `git pull`, which brings the new files of this class (the Fe input and pseudopotential) into the copy you cloned last week.

In [ ]:
# Quantum ESPRESSO from Ubuntu's package manager (about 1 min).
!apt-get update -qq
!apt-get install -y -qq quantum-espresso > /dev/null

# Ubuntu 24.04 fix (see Aula 6): without it pw.x stops with "*** buffer overflow detected ***".
!curl -sL https://raw.githubusercontent.com/flavianojs/dft-cbpf/main/src/qe_fix.c -o /tmp/qe_fix.c
!gcc -O2 -shared -fPIC -o /usr/local/lib/qe_fix.so /tmp/qe_fix.c
!echo /usr/local/lib/qe_fix.so > /etc/ld.so.preload

# Today we use four programs of the package: pw.x, bands.x, dos.x, projwfc.x (and ld1.x).
# Expected output: five paths in /usr/bin.
!which pw.x bands.x dos.x projwfc.x ld1.x

In [ ]:
# Mount Google Drive (authorize when asked) and go to the course folder.
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive

REPO = "https://github.com/flavianojs/dft-cbpf.git"
# Clone only if the folder does not exist yet (first time) ...
![ -d dft-cbpf ] || git clone -q --depth 1 {REPO}
# ... and bring in what is new since then (today: src/fe/ and the Fe pseudopotential).
# Your own files in calculations/aula6/ are not touched.
!git -C dft-cbpf pull -q

%cd dft-cbpf/calculations
!ls ../src/fe ../src/pseudos

## 2. Helper functions
`set_var`, `set_kpts` and `etot` are the ones from Aula 6. Two changes:

* `run` now takes the program as an argument (`pw.x`, `bands.x`, `dos.x`, `projwfc.x`, `ld1.x`): the post-processing programs read an input file just like `pw.x`;
* `add_var` adds a variable that is not in the input yet (for example `occupations`).

Everything goes to `calculations/aula7/`.

In [ ]:
import re, subprocess, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def set_var(text, name, value):
    # Change the line  name = ...  of a namelist (first match, any case). See Aula 6.
    pat = rf"^(\s*{re.escape(name)}\s*=\s*).*$"
    text, n = re.subn(pat, rf"\g<1>{value}", text, count=1, flags=re.M | re.I)
    assert n == 1, f"{name} not found"
    return text


def add_var(text, name, value, namelist="&SYSTEM"):
    # Insert the line  name = value  right after the line that opens the namelist.
    # Use it for variables that are not in the input yet (set_var would stop with "not found").
    text, n = re.subn(rf"^(\s*{namelist}.*)$", rf"\g<1>\n  {name} = {value}", text,
                      count=1, flags=re.M | re.I)
    assert n == 1, f"{namelist} not found"
    return text


def set_kpts(text, mesh):
    # Replace the line after K_POINTS, e.g. "8 8 8 1 1 1". See Aula 6.
    return re.sub(r"(K_POINTS.*\n).*", rf"\g<1>  {mesh}", text, count=1)


def run(text, name, prog="pw.x", workdir="aula7"):
    # Write workdir/name.in, run   prog < name.in > name.out   inside workdir, return the output.
    w = Path(workdir)
    w.mkdir(exist_ok=True)
    (w / f"{name}.in").write_text(text)
    t0 = time.time()
    with open(w / f"{name}.in") as inp, open(w / f"{name}.out", "w") as out:
        subprocess.run([prog], stdin=inp, stdout=out, stderr=subprocess.STDOUT, cwd=w)
    out = (w / f"{name}.out").read_text()
    # QE programs end a successful run with "JOB DONE" (ld1.x: "End of All-electron run").
    # If it is missing, show the end of the output (where the error message is) and stop.
    if not re.search(r"JOB DONE|End of All-electron run", out):
        print(out[-1500:])
        raise RuntimeError(f"{prog} failed: see {workdir}/{name}.out")
    print(f"{prog:10s} {name:14s} done in {time.time() - t0:5.1f} s", flush=True)
    return out


def etot(out):
    # Converged total energy in Ry (the line that starts with "!").
    return float(re.findall(r"!\s+total energy\s+=\s+(\S+)", out)[-1])


RY_EV = 13.605693   # 1 Ry in eV

## 3. Silicon: SCF, then bands along a path
Three steps, all with the same `prefix` and `outdir`, so that each one reads what the previous one wrote:

1. `pw.x`, `calculation = 'scf'`: the self-consistent density, on the converged parameters of Aula 6 (40 Ry, shifted 8×8×8);
2. `pw.x`, `calculation = 'bands'`: no new SCF; the Hamiltonian built from that density is diagonalized at the k points of a path;
3. `bands.x`: collects the eigenvalues along the path into a file for plotting.

Expected: `E = -15.85306732 Ry`, the same energy as at a = 10.26 bohr in Aula 6.

In [ ]:
# Step 1: SCF with the parameters converged in Aula 6.
base = Path("../src/silicon/pw.scf.silicon.in").read_text()
inp = set_var(base, "pseudo_dir", "'../../src/pseudos'")   # relative to aula7/
inp = set_var(inp, "ecutwfc", 40)
inp = set_kpts(inp, "8 8 8 1 1 1")

out = run(inp, "si_scf")
print(f"E = {etot(out):.8f} Ry")

The path: `K_POINTS crystal_b` gives the corners in units of the reciprocal lattice vectors, each followed by the number of points to the next corner.
The 1 after U makes the path jump from U to K: the two points are equivalent, but the line K–Γ is a high-symmetry line and U–Γ is not.
`nbnd = 8` (already in the input): 4 occupied bands (8 electrons) and 4 empty ones.

In [ ]:
# Step 2: bands along L - Gamma - X - U|K - Gamma.
PATH = '''K_POINTS crystal_b
6
  0.000  0.500  0.000  30  ! L
  0.000  0.000  0.000  40  ! G
 -0.500  0.000 -0.500  15  ! X
 -0.375  0.250 -0.375   1  ! U   (1: jump to K)
 -0.375  0.375  0.000  45  ! K
  0.000  0.000  0.000   1  ! G
'''
inp_b = set_var(inp, "calculation", "'bands'")
inp_b = re.sub(r"K_POINTS[\s\S]*", PATH, inp_b)   # replace the whole K_POINTS card

out = run(inp_b, "si_bands")
print(re.search(r"number of k points=\s*\d+", out).group())   # 30+40+15+1+45+1 = 132

In [ ]:
# Step 3: bands.x writes the bands along the path to si_bands.dat.gnu (for plotting).
pp = '''&BANDS
  prefix  = 'silicon'
  outdir  = './tmp/'
  filband = 'si_bands.dat'
/
'''
out = run(pp, "pp_bands", prog="bands.x")

# bands.x also prints the position of each corner of the path along the x axis of the plot.
xhs = [float(x) for x in re.findall(r"high-symmetry point:.*x coordinate\s+(\S+)", out)]
print("corners at x =", xhs)          # U and K get the same x: bands.x detects the jump

### Plot and read the bands
Expected: valence-band maximum (VBM) at Γ; conduction-band minimum (CBM) on the Γ–X line, at about 85% of the way to X;
indirect gap 0.52 eV (experiment: 1.17 eV); direct gap at Γ 2.56 eV (experiment: ≈ 3.4 eV); valence band 11.9 eV wide (experiment: ≈ 12.5 eV).

In [ ]:
# The .gnu file has one block per band (columns: x, energy in eV), blocks separated by a blank line.
blocks = Path("aula7/si_bands.dat.gnu").read_text().strip().split("\n\n")
bands = np.array([np.loadtxt(b.splitlines()) for b in blocks])   # shape (nbnd, nk, 2)
x, e = bands[0, :, 0], bands[:, :, 1]                             # x: (nk,), e: (nbnd, nk)

E_VBM = e[3].max()     # band 4 (index 3) is the highest occupied band
E_CBM = e[4].min()     # band 5 is the lowest empty band
iG = np.argmin(abs(x - xhs[1]))                                   # Gamma, second corner
xc = x[np.argmin(e[4])]                                           # where the CBM is
print(f"VBM at x = {x[np.argmax(e[3])]:.3f} (Gamma is at {xhs[1]:.3f})")
print(f"CBM at x = {xc:.3f}: {(xc - xhs[1]) / (xhs[2] - xhs[1]):.2f} of the way from Gamma to X")
print(f"indirect gap  = {E_CBM - E_VBM:.2f} eV   (experiment 1.17 eV)")
print(f"direct gap at Gamma = {e[4, iG] - e[3, iG]:.2f} eV   (experiment ~3.4 eV)")
print(f"valence band width  = {E_VBM - e[0].min():.2f} eV  (experiment ~12.5 eV)")

for band in e:
    plt.plot(x, band - E_VBM, color="C0")
for xv in xhs:
    plt.axvline(xv, color="gray", lw=0.5)
plt.axhline(0, color="gray", ls=":")
# Corner labels. U and K sit at the same x (the jump), so they share one tick: "U|K".
ticks = {}
for xv, lab in zip(xhs, ["L", "Γ", "X", "U", "K", "Γ"]):
    ticks[xv] = ticks[xv] + "|" + lab if xv in ticks else lab
plt.xticks(list(ticks), list(ticks.values()))
plt.xlim(x[0], x[-1])
plt.ylabel("ε − ε_VBM (eV)")
plt.title("Silicon, LDA")
plt.show()

## 4. Silicon: density of states (DOS)
Bands along lines are not enough for a DOS: we need eigenvalues on a dense uniform mesh of the whole Brillouin zone.
`calculation = 'nscf'` does that (again without a new SCF) and also computes the occupations, which `dos.x` and `projwfc.x` need.

`occupations = 'tetrahedra_opt'`: the tetrahedron method interpolates the bands linearly between mesh points and integrates the delta functions analytically, with no broadening parameter.
The 16×16×16 mesh has 145 irreducible points; this run takes about 20–40 s.

In [ ]:
inp_n = set_var(inp, "calculation", "'nscf'")
inp_n = set_var(inp_n, "nbnd", 12)                                # a few more empty bands
inp_n = add_var(inp_n, "occupations", "'tetrahedra_opt'")         # not in the input yet
inp_n = set_kpts(inp_n, "16 16 16 0 0 0")                         # dense mesh, with Gamma

out = run(inp_n, "si_nscf")
print(re.search(r"number of k points=.*", out).group())

`dos.x` twice on the same nscf: once with tetrahedra (the default after a tetrahedra run), once with a Gaussian broadening of 0.2 eV (`degauss` is in Ry).
Expected: the integrated DOS up to the VBM is 8.00 electrons per cell.

In [ ]:
def dos_input(fildos, extra=""):
    return f'''&DOS
  prefix = 'silicon'
  outdir = './tmp/'
  fildos = '{fildos}'
  DeltaE = 0.02
  Emin = -7.0
  Emax = 12.0
{extra}/
'''

run(dos_input("si_dos_tetra.dat"), "pp_dos_tetra", prog="dos.x")
run(dos_input("si_dos_gauss.dat", "  degauss = 0.0147\n"), "pp_dos_gauss", prog="dos.x")

# Columns of the dos.x file: energy (eV), DOS (states/eV/cell), integrated DOS.
d_t = np.loadtxt("aula7/si_dos_tetra.dat")
d_g = np.loadtxt("aula7/si_dos_gauss.dat")
print(f"electrons below the VBM: {np.interp(E_VBM, d_t[:, 0], d_t[:, 2]):.2f}")

plt.plot(d_g[:, 0] - E_VBM, d_g[:, 1], label="Gaussian, 0.2 eV")
plt.plot(d_t[:, 0] - E_VBM, d_t[:, 1], label="tetrahedra")
plt.axvline(0, color="gray", ls=":")
plt.xlabel("ε − ε_VBM (eV)"); plt.ylabel("DOS (states/eV/cell)")
plt.legend(); plt.title("Silicon, 16×16×16"); plt.show()

### Projected DOS and Löwdin charges
`projwfc.x` projects every Bloch state on the atomic orbitals stored in the pseudopotential (3s and 3p for Si).
Expected: Löwdin charges per atom s = 1.14 and p = 2.82, against 3s² 3p² in the free atom (about one electron goes from s to p: sp³ bonding); spilling parameter 0.0093, below 1%.

In [ ]:
proj = '''&PROJWFC
  prefix  = 'silicon'
  outdir  = './tmp/'
  filpdos = 'si'
  DeltaE  = 0.02
  Emin = -7.0
  Emax = 12.0
/
'''
out = run(proj, "pp_projwfc", prog="projwfc.x")

# Loewdin charges of each atom and the spilling parameter, as printed by projwfc.x.
for line in out.splitlines():
    if line.lstrip().startswith(("Atom #", "Spilling")):
        print(line.strip())

# One file per atom and orbital: si.pdos_atm#1(Si)_wfc#1(s), ... Column 2 is the projected DOS.
pd = {}
for l in ("s", "p"):
    files = sorted(Path("aula7").glob(f"si.pdos_atm#*_wfc#*({l})"))
    pd[l] = sum(np.loadtxt(f)[:, 1] for f in files)              # sum over the two atoms
E = np.loadtxt("aula7/si.pdos_tot")[:, 0]
plt.plot(d_t[:, 0] - E_VBM, d_t[:, 1], color="gray", label="total")
plt.plot(E - E_VBM, pd["s"], label="Si s")
plt.plot(E - E_VBM, pd["p"], label="Si p")
plt.axvline(0, color="gray", ls=":")
plt.xlabel("ε − ε_VBM (eV)"); plt.ylabel("states/eV/cell")
plt.legend(); plt.show()

## 5. What is a Kohn–Sham eigenvalue? The silicon atom
`ld1.x` solves the Kohn–Sham equations of a free atom (all electrons, spherical symmetry). We change the occupation f of the 3p orbital from 2 to 1 and look at
the total energy E(f) and the eigenvalue ε_3p(f).

Janak's theorem says ε_3p = dE/df. The ionization energy is the finite difference I = E(f=1) − E(f=2).
Expected: −ε_3p of the neutral atom is 4.18 eV, but I = 7.84 eV (experiment 8.15 eV); −ε_3p at f = 1.5 is 7.77 eV, close to I. A few seconds.

In [ ]:
fs = np.arange(2.0, 0.99, -0.25)        # occupation of 3p: 2, 1.75, 1.5, 1.25, 1
E_at, eps = [], []
for f in fs:
    text = f'''&input
  title = 'Si', zed = 14.0, iswitch = 1, dft = 'PZ',
  config = '[Ne] 3s2 3p{f:.2f}'
/
'''
    out = run(text, f"si_atom_{f:.2f}", prog="ld1.x")
    E_at.append(float(re.search(r"Etot\s*=.*?(\S+)\s+eV", out).group(1)))          # eV
    eps.append(float(re.search(r"^\s*3 1\s+3P.*\s(\S+)\s*$", out, re.M).group(1)))  # eV
E_at, eps = np.array(E_at), np.array(eps)

I = E_at[-1] - E_at[0]
print(f"-eps_3p (neutral, f = 2)     = {-eps[0]:.2f} eV")
print(f"-eps_3p (half-way, f = 1.5)  = {-eps[2]:.2f} eV")
print(f"I = E(Si+) - E(Si)           = {I:.2f} eV   (experiment 8.15 eV)")
print("Janak check, dE/df vs eps:", np.round(np.gradient(E_at, fs)[1:-1], 2), np.round(eps[1:-1], 2))

plt.plot(fs, E_at - E_at[0], "o-", label="E(f), LDA")
plt.plot([2, 1], [0, I], "--", label="straight line (exact functional)")
plt.xlabel("occupation of 3p"); plt.ylabel("E − E(f=2) (eV)")
plt.legend(); plt.show()

## 6. Iron: is bcc Fe magnetic?
The input `src/fe/pw.scf.fe.in` (open it in the file panel and read the comments) is ferromagnetic bcc Fe: PBE, ultrasoft pseudopotential with 3s and 3p in the valence (16 electrons),
cold smearing, `nspin = 2` and a starting magnetization of 0.6.

Expected (45 Ry): total magnetization 2.34 μB, absolute 2.49 μB; the non-magnetic solution is 651 meV/atom higher. About 30 s.

In [ ]:
fe = Path("../src/fe/pw.scf.fe.in").read_text()
fe = set_var(fe, "pseudo_dir", "'../../src/pseudos'")

def mag(out):
    # Last values of the total and absolute magnetization (Bohr magnetons per cell).
    tot = float(re.findall(r"total magnetization\s+=\s+(\S+)", out)[-1])
    ab = float(re.findall(r"absolute magnetization\s+=\s+(\S+)", out)[-1])
    return tot, ab

out_fm = run(fe, "fe_fm")
E_fm = etot(out_fm)
print(f"FM: E = {E_fm:.8f} Ry,  M = {mag(out_fm)[0]:.2f} muB,  |M| = {mag(out_fm)[1]:.2f} muB")
# Moment inside a sphere around the atom (depends on the sphere radius: an estimate).
print(re.findall(r"atom:\s+1\s+charge:.*", out_fm)[-1])

# Non-magnetic: nspin = 1 (starting_magnetization is then ignored).
# A different prefix keeps the files of the FM run in tmp/fe.save: the DOS at the end reads them.
fe_test = set_var(fe, "prefix", "'fe_test'")
out_nm = run(set_var(fe_test, "nspin", 1), "fe_nm")
E_nm = etot(out_nm)
print(f"NM: E = {E_nm:.8f} Ry")
print(f"E(FM) - E(NM) = {(E_fm - E_nm) * RY_EV * 1000:.0f} meV/atom")

### The seed matters
With `starting_magnetization = 0` the up and down densities start equal, the two Kohn–Sham Hamiltonians are identical, and they stay equal:
the SCF converges to the non-magnetic solution even with `nspin = 2`. Any small seed is enough to break the symmetry.
Expected: seed 0 → M = 0 and the NM energy; seed 0.05 → the FM solution.

In [ ]:
for seed in (0.0, 0.05):
    out = run(set_var(fe_test, "starting_magnetization(1)", seed), f"fe_seed{seed}")
    print(f"seed {seed:4.2f}: M = {mag(out)[0]:5.2f} muB,  E - E(FM) = {(etot(out) - E_fm) * RY_EV * 1000:7.1f} meV")

### Convergence: the moment needs more than the energy
The pseudopotential header suggests 71 Ry. Expected: at 30 Ry, M = 2.81 μB and −899 meV/atom; at 45 and 60 Ry the energy difference is −651 meV/atom in both, but M goes from 2.34 to 2.29 μB.
About 2 min.

In [ ]:
print(" ecutwfc   M (muB)   E(FM)-E(NM) (meV/atom)")
for ec in (30, 45, 60):
    f = set_var(set_var(fe_test, "ecutwfc", ec), "ecutrho", 8 * ec)   # prefix fe_test
    o_fm = run(f, f"fe_fm_ec{ec}")
    o_nm = run(set_var(f, "nspin", 1), f"fe_nm_ec{ec}")
    print(f"   {ec:3d}     {mag(o_fm)[0]:5.2f}      {(etot(o_fm) - etot(o_nm)) * RY_EV * 1000:7.0f}")

### DOS by spin
An nscf run on a 16×16×16 mesh with tetrahedra, then `dos.x` (about 1 min in all). With `nspin = 2`, `dos.x` writes one column per spin.
Expected: E_F = 17.411 eV; the two curves have almost the same shape, shifted by the exchange splitting: d-band centers at −1.86 eV (up) and +0.07 eV (down), 1.93 eV apart.

In [ ]:
fe_n = set_var(fe, "calculation", "'nscf'")
fe_n = set_var(fe_n, "occupations", "'tetrahedra_opt'")
fe_n = add_var(fe_n, "nbnd", 20)
fe_n = set_kpts(fe_n, "16 16 16 0 0 0")
run(fe_n, "fe_nscf")

fe_dos = '''&DOS
  prefix = 'fe'
  outdir = './tmp/'
  fildos = 'fe_dos.dat'
  DeltaE = 0.05
  Emin = 9.0
  Emax = 22.0
/
'''
run(fe_dos, "pp_dos_fe", prog="dos.x")

# Columns: energy (eV), DOS up, DOS down, integrated DOS. The Fermi energy is in the header.
txt = Path("aula7/fe_dos.dat").read_text()
EF = float(re.search(r"EFermi\s*=\s*(\S+)", txt).group(1))
d = np.loadtxt("aula7/fe_dos.dat")
e = d[:, 0] - EF

# Center of the d bands of each spin (window -8 to 4 eV around E_F): the exchange splitting.
w = (e > -8) & (e < 4)
c_up = (e[w] * d[w, 1]).sum() / d[w, 1].sum()
c_dn = (e[w] * d[w, 2]).sum() / d[w, 2].sum()
print(f"E_F = {EF:.3f} eV;  d-band centers: up {c_up:.2f} eV, down {c_dn:.2f} eV;  splitting {c_dn - c_up:.2f} eV")

plt.fill_between(e, d[:, 1], alpha=0.3, label="spin up (majority)")
plt.fill_between(e, -d[:, 2], alpha=0.3, label="spin down (minority)")
plt.axvline(0, color="gray", ls=":"); plt.axhline(0, color="gray", lw=0.5)
plt.xlim(-8, 4)
plt.xlabel("ε − E_F (eV)"); plt.ylabel("DOS (states/eV/atom), down drawn negative")
plt.legend(); plt.show()

### For the mini-project
* Bands: `scf` → `bands` with a path (`crystal_b`) → `bands.x`. For other lattices, get the standard path from SeeK-path (seekpath.materialscloud.io).
* DOS: `scf` → `nscf` on a denser mesh with `tetrahedra_opt` → `dos.x` and `projwfc.x`.
* Do not read the gap from a coarse SCF mesh: find the band extrema on the path or on a dense mesh.
* Magnetic materials: `nspin = 2` and a seed; compare the energy with the non-magnetic (and, if relevant, antiferromagnetic) solution; converge the moment, not only the energy.

### Next session
Re-run sections 1 and 2, then the cell of section 3 that defines `inp` (and the first cell of section 6 for `fe`). Everything in `aula7/` is still in your Drive.